# Week 7 — Optimization

**Focus:** How do optimization and unsupervised representations behave in noisy, redundant data?

*Live-coding notebook*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smc77/uc_finmlai/blob/main/lectures/week07/week7_demos.ipynb)

## What this notebook is about

Many financial ML problems are hard not because the objective is mysterious, but because the data are noisy, redundant, and badly conditioned. This notebook asks how optimization and unsupervised representations behave under those conditions.

You will compare gradient-based optimizers on an ill-conditioned surface, watch small input errors become large portfolio changes, audit search budgets and future invariance, compress a factor panel with PCA, compare sample eigenvalues with a random-matrix noise band, and use clustering to organize assets and candidate regimes.

By the end, you should be able to:

- explain why scaling and conditioning affect optimization;
- distinguish numerical convergence from economic correctness;
- read a principal component as an estimated direction rather than a named truth;
- explain what an eigenvalue noise band is screening for; and
- treat clusters as descriptions whose stability and decision value still require testing.

The simulations make recovery possible or impossible by design. The industry-data companion shows what the same tools look like without supplying known labels.

### Course context

For a concise review and the optional textbook route, see [Week 7 summary](https://github.com/smc77/uc_finmlai/blob/main/lectures/week07/week7_summary.md) and [Week 7 reading guide](https://github.com/smc77/uc_finmlai/blob/main/lectures/week07/reading.md). The notebook itself is designed to remain understandable without them.

## How to use this notebook

At the first break, run any **Setup** cell and then the **Imports** cell. If there is no Setup cell, begin with Imports. After that, use the slide cue to jump to the named demo; you do not need to rerun the whole notebook at every break. To check the whole notebook from a clean start, use **Runtime → Run all**.

Each demo follows the same rhythm: predict what the output should show, run the cell, and follow the task immediately underneath it. An **After you run** cell is editable: double-click it, replace `[write here]`, and press Shift-Enter. When an editable research-record entry appears, its completed comparison sits below it in a collapsed box—write your version before opening that box. This notebook is generated from the same source code the lecturer runs live.

## Jump to a demonstration

Both links jump within *this* notebook — neither opens a new tab. Use **Colab** when the notebook is open in Colab, **Jupyter** when it is open in Jupyter or rendered to HTML. You do not need to scroll through or rerun the whole notebook.

> If the **Jupyter** links do nothing in JupyterLab or Notebook 7, that is the windowed renderer, not a broken link: cells outside the viewport are not in the page, so there is no anchor to scroll to. Set *Settings → Settings Editor → Notebook → Windowing mode* to `defer` (or `none`) and they work.

### Deck A

<ul>
<li>Demo 1 — Convex and non-convex objective geometry — <a href="#Demo-1-%E2%80%94-Convex-and-non-convex-objective-geometry">Jupyter</a> · <a href="#scrollTo=md-965ed26e7d57">Colab</a></li>
<li>Demo 2 — Gradient descent vs. momentum vs. Adam on an ill-conditioned bowl — <a href="#Demo-2-%E2%80%94-Gradient-descent-vs.-momentum-vs.-Adam-on-an-ill-conditioned-bowl">Jupyter</a> · <a href="#scrollTo=md-09006b9e5185">Colab</a></li>
<li>Demo 3 — Search budget and development overfitting — <a href="#Demo-3-%E2%80%94-Search-budget-and-development-overfitting">Jupyter</a> · <a href="#scrollTo=md-e3963dd4248d">Colab</a></li>
<li>Demo 4 — Stress a tiny mean-variance optimum — <a href="#Demo-4-%E2%80%94-Stress-a-tiny-mean-variance-optimum">Jupyter</a> · <a href="#scrollTo=md-0338f3db3208">Colab</a></li>
</ul>

### Deck B

<ul>
<li>Demo 5 — PCA on a simulated factor panel — <a href="#Demo-5-%E2%80%94-PCA-on-a-simulated-factor-panel">Jupyter</a> · <a href="#scrollTo=md-b44785d63714">Colab</a></li>
<li>Demo 6 — Future-invariance test for PCA — <a href="#Demo-6-%E2%80%94-Future-invariance-test-for-PCA">Jupyter</a> · <a href="#scrollTo=md-e1fe84657a0c">Colab</a></li>
<li>Real-data companion — PCA on twelve industry portfolios — <a href="#Real-data-companion-%E2%80%94-PCA-on-twelve-industry-portfolios">Jupyter</a> · <a href="#scrollTo=md-de33b3e667e3">Colab</a></li>
<li>Demo 7 — Marchenko-Pastur eigenvalue band — <a href="#Demo-7-%E2%80%94-Marchenko-Pastur-eigenvalue-band">Jupyter</a> · <a href="#scrollTo=md-7ea2c9043d5b">Colab</a></li>
<li>Demo 8 — Cluster a correlation matrix to recover sectors — <a href="#Demo-8-%E2%80%94-Cluster-a-correlation-matrix-to-recover-sectors">Jupyter</a> · <a href="#scrollTo=md-e118236be73e">Colab</a></li>
<li>Demo 9 — Frozen-centroid regime assignment — <a href="#Demo-9-%E2%80%94-Frozen-centroid-regime-assignment">Jupyter</a> · <a href="#scrollTo=md-ad14ad4926d2">Colab</a></li>
<li>Optional example — the yield curve, where the components have names — <a href="#Optional-example-%E2%80%94-the-yield-curve,-where-the-components-have-names">Jupyter</a> · <a href="#scrollTo=md-bc15d3453dd0">Colab</a></li>
</ul>

## Read the code: choose, fit, and check

Start with the question, then read the small calculation, then inspect the graph.
**Optimization** searches for a better value of a stated objective under stated
rules. **Unsupervised learning** estimates structure without a supplied prediction
target. Both still learn from a dated sample.

The nine numbered demos serve six lecture exercises. The industry companion belongs
to the PCA exercise; the final Treasury example is optional. Simulations use
`finmlsim` when its model matches the question, and NumPy for small constructions
whose geometry or exact truth needs to be visible.

## Project research-record checkpoint

Your project record remains the single `RESEARCH_RECORD.md` at the project-repository root; this notebook only supplies a checkpoint. After this week, update whichever of these fields actually apply:

1. **Fitted state:** transformations or representations, fit sample and cutoff, optimizer or solver, and version.
2. **Search ledger:** candidate representations, hyperparameters or cluster counts, selection rule, and total trials.
3. **Stability:** conditioning, perturbation, resampling, or later-period evidence that could overturn the choice.
4. **Baseline:** the simpler feasible representation or allocation against which the added machinery is evaluated.
5. **Interpretation limit:** what an estimated component, cluster, or regime does not establish.

If a choice changes after seeing development evidence, append a dated decision-log entry; do not silently rewrite the earlier plan.

### Six recorded-lecture mini-projects

1. **Classify geometry — Demo 1:** use the chord test to distinguish a convex objective from a constructive non-convex counterexample.
2. **Make descent crawl — Demo 2:** compare optimizer paths on ill-conditioned geometry and alter the scaling.
3. **Cap the search — Demo 3:** hold the candidate sequence fixed and compare development with untouched loss at three budgets.
4. **Stress the optimum — Demo 4:** perturb means and covariance and record weight movement, turnover, and objective regret.
5. **Audit a PCA representation — Demos 5–7 plus the real-data PCA companion:** interpret a planted factor, verify future invariance, compare eigenvalues with a noise band, and test a frozen component on industry returns.
6. **Recover groups, then make a regime prospective — Demos 8–9:** recover planted asset sectors, then freeze training-period centroids and compare later assignments with retrospective full-sample labels.

In [ ]:
# Setup: install packages not preinstalled in Colab.
%pip install -q finmlsim

### Imports

> **Run this once before any demo.** It loads the packages and helper functions used below; there is no result to interpret in this cell.

In [ ]:
from io import BytesIO, StringIO
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from scipy.linalg import eigh

from finmlsim import simulate as sim

rng = np.random.default_rng(42)


def course_csv(relative_path):
    """Read bundled Fama-French data locally, or its public source in Colab."""
    for root in (Path.cwd(), *Path.cwd().parents):
        local = root / relative_path
        if local.exists():
            return pd.read_csv(local), str(local)
    archives = {
        "datasets/famafrench/ff_factors_daily.csv": "F-F_Research_Data_Factors_daily_CSV.zip",
        "datasets/famafrench/ff_12industry_daily.csv": "12_Industry_Portfolios_daily_CSV.zip",
    }
    archive = archives[relative_path]
    url = f"https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/{archive}"
    with urlopen(url) as response, ZipFile(BytesIO(response.read())) as zipped:
        lines = zipped.read(zipped.namelist()[0]).decode("utf-8").splitlines()
    header_row = next(i for i, line in enumerate(lines) if line.startswith(","))
    rows = []
    for line in lines[header_row + 1:]:
        first = line.split(",", 1)[0].strip()
        if len(first) == 8 and first.isdigit():
            rows.append(line)
        elif rows:
            break
    frame = pd.read_csv(StringIO("\n".join([lines[header_row], *rows])))
    return frame.rename(columns={frame.columns[0]: "date"}), url


def save_lecture_figure(fig, filename):
    """Save a notebook figure when the course repository is available."""
    for root in (Path.cwd(), *Path.cwd().parents):
        output = root / "lectures" / "week07" / "figures"
        if output.exists():
            target = output / filename
            fig.savefig(target, dpi=180, bbox_inches="tight", facecolor="white")
            print(f"saved lecture figure: {target}")
            return

### Demo 1 — Convex and non-convex objective geometry

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

### 1. Does the curve stay below its joining line?

**Question:** which objective is convex? The input is one candidate decision `x`;
`objective(x)` returns its score. `np.linspace` supplies evenly spaced candidates.
The code evaluates two explicit functions, not fitted models.

A chord is the straight line between two function values. For `f(x)=x**2`, the
points at −1 and +1 both have height 1, while the midpoint has height 0: the curve
lies below that chord. `chord_y - objective(chord_x)` calculates this vertical gap
at 150 locations. A negative gap supplies a counterexample to convexity.
Checking one chord cannot prove that every chord passes; the bowl's convexity
also follows from its known formula.

**Read the output:** horizontal position is the decision; height is the objective.
The dashed segment is the chord, not a fitted regression line.

**Predict, change, explain:** choose two points in one valley of the double-well
function. Does that chord still reveal non-convexity? Explain why a passing local
picture does not prove a global property. Then classify the four lecture problems,
including their constraints and whether the optimum must be unique.

In [ ]:
x_grid = np.linspace(-1.5, 1.5, 400)

def convex_objective(x):
    return x ** 2

def nonconvex_objective(x):
    return (x ** 2 - 1) ** 2

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
examples = [
    (convex_objective, (-1.2, 1.0), "convex objective"),
    (nonconvex_objective, (-1.15, 1.15), "non-convex objective"),
]
for ax, (objective, (left, right), title) in zip(axes, examples):
    chord_x = np.linspace(left, right, 150)
    chord_y = (objective(left) +
               (objective(right) - objective(left)) *
               (chord_x - left) / (right - left))
    gap = chord_y - objective(chord_x)
    ax.plot(x_grid, objective(x_grid), lw=2.4, label="objective")
    ax.plot(chord_x, chord_y, "--", lw=2.0, label="chord")
    ax.scatter([left, right], [objective(left), objective(right)], zorder=3)
    ax.set_title(title)
    ax.set_xlabel("decision variable")
    ax.set_ylabel("objective")
    ax.legend(frameon=False)
    print(f"{title}: minimum(chord - objective) = {gap.min():+.3f}")
fig.suptitle("A convex objective stays below every chord; a non-convex one need not")
fig.tight_layout()
save_lecture_figure(fig, "demo1_convex_nonconvex.png")
plt.show()

print("A negative chord gap is a constructive failure of convexity.")
print("This geometric test says nothing about whether the objective is economically well specified.")

### Mini-project: classify the problem

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

For four objectives, identify the variable, feasible set, convexity,
and whether the solution is unique.

**Notebook:** run **Demo 1 — Convex and non-convex objective geometry**, then classify the four problems.

| Problem | Variable | Feasible set | Convex? | Unique solution? |
|---|---|---|---|---|
| Ridge regression | | | | |
| Lasso regression | | | | |
| Long-only mean-variance | | | | |
| Integer lot sizes | | | | |

**Research record:** for each of the four objectives: the variable chosen, the feasible set, whether the problem is convex, and whether the solution is unique.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 2 — Gradient descent vs. momentum vs. Adam on an ill-conditioned bowl

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

### 2. Why can changing units slow the same calculation?

**First question: how do updates differ?** `w` has shape `(2,)`. The objective is
`0.5*(w[0]**2 + 25*w[1]**2)` and `grad(w)` is `[w[0],25*w[1]]`.
At `[1,1]`, a gradient-descent step of .05 gives `[.95,-.25]`.
`run` stores every parameter vector with `.copy()` so earlier history is preserved.
Momentum accumulates gradients; Adam averages gradients and squared gradients,
corrects their initial bias, and divides by their running scale. The declared
rates are .05, .05, and .30; this is not a fair tuning contest or a universal ranking.

**Second question: does rescaling change the prediction problem?** `X_train` has
160 rows and two columns. QR creates orthogonal centered columns so units are the
main source of poor conditioning. Multiplying column 2 by 20 leaves an unpenalized
linear prediction family unchanged: divide its coefficient by 20 to compensate.
The quadratic curvature in that coordinate grows by 400. The original rate .2
therefore diverges; .004 works slowly. `StandardScaler.fit` sees training rows only.
The centered response plus saved `y_mean` supplies the intercept.

`fit_descent` computes `X.T @ residual / len(X)` and stops when loss is within
`1e-10` of an independently computed least-squares minimum, when it diverges, or
when the budget ends. Later rows check prediction agreement, not tuning or accuracy.
The three panels show parameter paths, objective by update, and the scaling
comparison's loss gap on log axes. Both loss and update axes in panel C are logarithmic.

**Predict, change, explain:** set the unit multiplier to 10 or change a step size.
Predict which run will fail before executing it. Compare statuses, update counts,
and predictions after convergence, rather than raw coefficients in different units.
This equivalence need not hold if a coefficient penalty is left unchanged.

Optional visual: [Grant Sanderson's gradient-descent lesson](https://www.3blue1brown.com/lessons/gradient-descent/).
Use the downhill-surface portion after the worked update above; the full neural-network
setup is not a prerequisite for this lab. Pause before a step and predict the direction.
The local runnable example supplies the lesson if the external player is unavailable.

In [ ]:
# An elongated quadratic: f(w) = 0.5 * (w1^2 + 25 * w2^2)
def f(w):
    return 0.5 * (w[0] ** 2 + 25 * w[1] ** 2)

def grad(w):
    return np.array([w[0], 25 * w[1]])

def run(opt, eta=0.05, T=80):
    w = np.array([1.0, 1.0])
    m = np.zeros(2)
    v = np.zeros(2)
    path = [w.copy()]
    for t in range(1, T + 1):
        g = grad(w)
        if opt == "gd":
            w = w - eta * g
        elif opt == "momentum":
            m = 0.9 * m + g
            w = w - eta * m
        elif opt == "adam":
            m = 0.9 * m + 0.1 * g
            v = 0.999 * v + 0.001 * g ** 2
            mhat = m / (1 - 0.9 ** t)
            vhat = v / (1 - 0.999 ** t)
            w = w - eta * mhat / (np.sqrt(vhat) + 1e-8)
        path.append(w.copy())
    return np.array(path)

optimizer_rates = {"gd": 0.05, "momentum": 0.05, "adam": 0.30}
paths = {name: run(name, eta=rate) for name, rate in optimizer_rates.items()}
# A second comparison changes feature units in one unpenalized regression.
# QR produces orthogonal centered training columns, so the scaling effect is visible.
scale_rng = np.random.default_rng(702)
raw_train = scale_rng.normal(size=(160, 2))
Q, _ = np.linalg.qr(raw_train - raw_train.mean(axis=0))
X_train = Q * np.sqrt(len(Q))
y_train = X_train @ np.array([1.0, -0.5]) + scale_rng.normal(0, .1, len(Q))
X_later = scale_rng.normal(size=(80, 2))  # prediction agreement only; no tuning
unit_factors = np.array([1.0, 20.0])
X_units = X_train * unit_factors
fitted_scaler = StandardScaler().fit(X_units)
X_scaled = fitted_scaler.transform(X_units)
y_mean = y_train.mean()
y_centered = y_train - y_mean

def fit_descent(X, target, eta, max_steps=20000, tolerance=1e-10):
    """Stop at a common objective-gap tolerance against a least-squares reference."""
    reference = np.linalg.lstsq(X, target, rcond=None)[0]
    minimum = np.mean((X @ reference - target) ** 2) / 2
    coefficients = np.zeros(X.shape[1])
    losses, coefficient_path = [], []
    status = "budget exhausted"
    for step in range(max_steps + 1):
        residual = X @ coefficients - target
        loss = np.mean(residual ** 2) / 2
        losses.append(loss); coefficient_path.append(coefficients.copy())
        if not np.isfinite(loss) or loss > 1e8:
            status = "diverged"; break
        if loss - minimum <= tolerance:
            status = "converged"; break
        if step < max_steps:
            coefficients -= eta * (X.T @ residual) / len(X)
    return {"coef": coefficients, "path": np.array(coefficient_path),
            "loss": np.array(losses), "minimum": minimum, "status": status,
            "steps": step}

scaling_runs = {
    "original units, rate .2": fit_descent(X_train, y_centered, .2),
    "20x units, same rate": fit_descent(X_units, y_centered, .2),
    "20x units, rate .004": fit_descent(X_units, y_centered, .004),
    "training-scaled, rate .2": fit_descent(X_scaled, y_centered, .2),
}
reference_coef = np.linalg.lstsq(X_train, y_centered, rcond=None)[0]
reference_prediction = y_mean + X_later @ reference_coef
scaling_prediction_errors = {}
for name, fit in scaling_runs.items():
    if fit["status"] == "converged":
        if name.startswith("training-scaled"):
            prediction = y_mean + fitted_scaler.transform(X_later * unit_factors) @ fit["coef"]
        else:
            prediction = y_mean + (X_later * (unit_factors if name.startswith("20x") else 1)) @ fit["coef"]
        scaling_prediction_errors[name] = np.max(np.abs(prediction - reference_prediction))
    print(f"{name}: {fit['status']}, {fit['steps']} updates")
print("Maximum later-prediction differences from least squares:", scaling_prediction_errors)

fig, axes = plt.subplots(1, 3, figsize=(12.6, 3.8))
x1, x2 = np.meshgrid(np.linspace(-1.15, 1.15, 180), np.linspace(-1.15, 1.15, 180))
axes[0].contour(x1, x2, 0.5 * (x1 ** 2 + 25 * x2 ** 2), levels=18,
               colors="#c7c7c7", linewidths=.7)
for name, path in paths.items():
    axes[0].plot(path[:, 0], path[:, 1], "-o", ms=2, label=name)
    axes[1].semilogy([max(f(w), 1e-12) for w in path], label=name)
axes[0].plot(0, 0, "k*", ms=10)
axes[0].set(xlabel="parameter 1", ylabel="parameter 2", title="A. Paths on the same bowl")
axes[1].set(xlabel="update", ylabel="objective (log scale)", title="B. Check progress numerically")
for name, fit in scaling_runs.items():
    gaps = np.maximum(fit["loss"] - fit["minimum"], 1e-12)
    axes[2].plot(np.arange(len(gaps)) + 1, gaps, label=name)
axes[2].set(xlabel="update + 1 (log scale)", ylabel="loss above optimum (log scale)",
            xscale="log", yscale="log", title="C. Same regression, different units")
for ax in axes:
    ax.legend(frameon=False, fontsize=7); ax.grid(alpha=.15)
fig.tight_layout()
save_lecture_figure(fig, "demo2_optimizer_paths.png")
plt.show()

### Mini-project: make gradient descent crawl

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

Run the supplied original-unit, rescaled, slow-rate, and training-scaled fits.
Change one scale or rate; compare loss, convergence status, and later predictions.

**Notebook:** run **Demo 2 — Gradient descent on an ill-conditioned bowl** and modify its scaling.

**Research record:** the scale and rates; convergence status and updates to the common tolerance; prediction agreement after convergence.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 3 — Search budget and development overfitting

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

### 3. Can a bigger search fit the measurement noise?

**Question:** what happens when we keep the best of more noisy candidates?
`theta` is a fixed array of 500 candidates. The known population objective is
`.20 + (theta-.70)**2`, minimized at .70. Development measurements add noise with
standard deviation .30. These are noisy objective estimates: negative values are
possible and are not negative squared errors.

For each prespecified budget, `argmin(development_loss[:budget])` stores an index.
Only after all indices are fixed do we generate the independent audit measurements,
with noise .05. Indexing those measurements with the stored indices tests the actual
choices. We never use `argmin(untouched_loss)` to select them.

**Read the output:** a falling development line is guaranteed by retaining the best
candidate so far. An improving independent line is not. This is one seeded measurement
experiment, not a fitted time-series backtest or a Monte Carlo result.

**Predict, change, explain:** rerun with another seed and explain why the audit line
need not worsen monotonically. If you use that line to choose a budget, it becomes
validation evidence and you need a new final test. Report all three prespecified budgets.

In [ ]:
search_rng = np.random.default_rng(1)
n_candidates = 500
theta = search_rng.uniform(-2, 2, n_candidates)
population_loss = 0.20 + (theta - 0.70) ** 2
development_loss = population_loss + search_rng.normal(0, 0.30, n_candidates)
budgets = np.array([20, 100, 500])
selected = [np.argmin(development_loss[:budget]) for budget in budgets]
best_development = np.array([development_loss[index] for index in selected])
# Generate independent audit outcomes only after every budget has selected.
untouched_loss = population_loss + search_rng.normal(0, 0.05, n_candidates)
selected_untouched = np.array([untouched_loss[index] for index in selected])

fig, ax = plt.subplots(figsize=(7.2, 3.8))
ax.plot(budgets, best_development, "o-", lw=2.2, label="best development loss")
ax.plot(budgets, selected_untouched, "o-", lw=2.2,
        label="untouched loss of selected candidate")
ax.set(xlabel="candidate evaluations", ylabel="noisy objective estimate",
       title="A larger search can fit development noise")
ax.legend(frameon=False); ax.grid(alpha=.2)
fig.tight_layout()
save_lecture_figure(fig, "demo3_search_budget.png")
plt.show()

for budget, index in zip(budgets, selected):
    print(f"budget {budget:3d}: theta={theta[index]:+.3f}, "
          f"development={development_loss[index]:+.3f}, "
          f"untouched={untouched_loss[index]:+.3f}")
print("The budgets and random-search sequence were fixed before the untouched losses were inspected.")
print("These are noisy objective measurements, not nonnegative squared errors; negative estimates are allowed.")
print("Comparing test outcomes is an audit of prespecified budgets, not a rule for selecting a budget on the final test.")

### Mini-project: cap the search

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

Run the same derivative-free optimizer at three evaluation budgets.
Plot best development objective and untouched objective against budget.

**Notebook:** run **Demo 3 — Search budget and development overfitting**.

**Research record:** the three evaluation budgets; best development objective at each; the untouched objective at each; the gap between them.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 4 — Stress a tiny mean-variance optimum

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

### 4. A precise answer can still be sensitive

**Question:** how far do investment weights move when estimated inputs move?
`mu` has 12 entries and `Sigma` is a 12-by-12 positive-definite covariance matrix.
The objective is `0.5*w @ Sigma @ w - mu @ w`; setting its gradient to zero gives
`Sigma @ w = mu`. `np.linalg.solve` solves that equation without explicitly forming
an inverse. In one dimension, mean .005 and variance .01 give weight .5.

The baseline is unconstrained: weights need not sum to one or be positive.
`cov_direction = A @ A.T / K` is positive semidefinite, so adding nonnegative
multiples preserves positive definiteness. The stress scalar also multiplies a
fixed mean-error direction. A scale of .20 is not a uniform 20% error in every input.

`norm(weights_i-base)` measures Euclidean weight movement. `abs(...).sum()` is the
sum of absolute target changes for one hypothetical rebalance, not annual turnover.
Regret is the stressed objective at the old weights minus its value at the new
optimum. These panels have different units; their visual heights are not comparable.
For a quadratic, regret also equals half the covariance-weighted squared movement.

**Predict, change, explain:** reduce the baseline correlation or alter the prespecified
stress direction. Predict the weight response, then check the residual
`covariance_i @ weights_i - mean_i`. This demo has no realized returns, financing,
trading costs, or binding constraints, so it cannot establish later profitability.

In [ ]:
# Show that plausible perturbations to both mu and Sigma can move w = inv(Sigma) @ mu.
K = 12
rho = 0.84
Sigma = rho * np.ones((K, K)) + (1 - rho) * np.eye(K)
mu = rng.standard_normal(K) * 0.02
stress_rng = np.random.default_rng(704)
mean_direction = stress_rng.standard_normal(K)
cov_direction = stress_rng.standard_normal((K, K))
cov_direction = cov_direction @ cov_direction.T / K
stress_levels = np.array([0.00, 0.05, 0.10, 0.20])

def mv_weights(mean, covariance):
    return np.linalg.solve(covariance, mean)

def mv_objective(weights, mean, covariance):
    return 0.5 * weights @ covariance @ weights - mean @ weights

base = mv_weights(mu, Sigma)
stressed_weights, instability, turnover, regret = [], [], [], []
for level in stress_levels:
    mean_i = mu + level * mu.std() * mean_direction
    covariance_i = Sigma + level * cov_direction
    weights_i = mv_weights(mean_i, covariance_i)
    stressed_weights.append(weights_i)
    instability.append(np.linalg.norm(weights_i - base))
    turnover.append(np.abs(weights_i - base).sum())
    regret.append(mv_objective(base, mean_i, covariance_i) -
                  mv_objective(weights_i, mean_i, covariance_i))

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
for ax, values, title in zip(
        axes, [instability, turnover, regret],
        ["weight instability", "absolute target change", "objective regret of base"]):
    ax.plot(stress_levels, values, "o-", lw=2.1)
    ax.set(xlabel="declared perturbation scale", title=title)
    ax.set_ylabel("modeled objective units" if "regret" in title else "weight units")
    ax.grid(alpha=.2)
fig.suptitle("Demo 4: nearby inputs can imply different portfolio actions")
fig.tight_layout(rect=(0, 0, 1, .91))
save_lecture_figure(fig, "demo4_portfolio_stress.png")
plt.show()

for level, move, trade, loss in zip(stress_levels, instability, turnover, regret):
    print(f"stress scale {level:.2f}: L2 move={move:.3f}, target change={trade:.3f}, regret={loss:.5f}")
print("Unconstrained weights: no sum-to-one or long-only rule, no realized returns or trading costs.")
print("The stress scalar multiplies declared directions; it is not a percentage error in every entry.")

### Mini-project: stress the optimum

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

Perturb expected returns and covariance within a declared range. Plot
weight instability, turnover, and the change in the modeled objective.

**Notebook:** run **Demo 4 — Stress a tiny mean-variance optimum**.

**Research record:** the declared perturbation range for means and covariance; weight instability, turnover, and the modeled-objective change across it.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 5 — PCA on a simulated factor panel

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

### 5. Which basket captures the most variation?

**Question:** does a deliberately planted shared movement create a broad first component?
`sim.panel` generates 1,500 dates by 60 assets as decimal log returns. Subtracting column means centers
returns; it does **not** divide by standard deviations. This is covariance PCA.
`PCA(n_components=10).fit(R)` learns ten unit-length directions from the 60-column
matrix. `components_[0]` has 60 entries; `R @ v1` gives one score per date.
A two-asset direction `[1/sqrt(2),1/sqrt(2)]` adds the centered returns and divides
by `sqrt(2)`; PCA learns its own weights instead.

The variance shares use total variance across all 60 directions, even though the
plot shows ten bars. Here PC1 explains about 31%, a large single share but not a
majority. Flipping a loading and its scores changes only the sign convention.
The correlation is with the **same-panel equal-weight average log return**, not the hidden
factor. That is useful supporting evidence but is not an independent market test.

**Predict, change, explain:** increase `idio_vol`, keeping the common-factor scale
fixed. Predict the first variance share. Compare with standardizing columns before
fitting, and explain why that answers a different question. High explained variance
does not establish a forecast of later returns.

In [ ]:
panel = sim.panel(n=1500, n_assets=60, market_vol=0.01, idio_vol=0.015, seed=7)
# sim.panel returns returns; center only (covariance PCA, not correlation PCA).
R = np.asarray(panel)
R = R - R.mean(0)

pca = PCA(n_components=10).fit(R)
print("Explained variance ratios (top 10):")
print(pca.explained_variance_ratio_.round(3))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar(range(1, 11), pca.explained_variance_ratio_[:10])
axes[0].set_title("Scree plot — one tall bar, then scree")
axes[0].set_xlabel("Component")
axes[0].set_ylabel("share of total variance")

# PC1 loadings
v1 = pca.components_[0]
if v1.mean() < 0:
    v1 = -v1
axes[1].bar(range(len(v1)), v1)
axes[1].set_title("PC1 loadings — same sign across the cross-section")
axes[1].set_xlabel("asset")
axes[1].set_ylabel("loading")
fig.suptitle("Demo 5: one planted common factor dominates the panel")
fig.tight_layout(rect=(0, 0, 1, .91))
save_lecture_figure(fig, "demo5_pca_factor_panel.png")
plt.show()

# PC1 score vs equal-weighted "market"
score1 = R @ v1
mkt = R.mean(axis=1)
corr = np.corrcoef(score1, mkt)[0, 1]
print(f"corr(PC1 score, EW market) = {corr:+.4f}  (expect ~0.99+)")

### Demo 6 — Future-invariance test for PCA

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

### 6. Can changing the future rewrite an earlier score?

**Question:** does a pipeline honor its training cutoff? There are 220 early rows
and 180 later rows, each with two features. `rotated_cloud` draws an elongated
Gaussian cloud and rotates it. The second history changes only the later block.

`protected_pca_fit(history, cutoff)` fits on `history[:cutoff]` and scores those
protected rows. Rebuilding it independently for both histories should give the
same answers. In contrast, fitting PCA to `vstack([early,later])` lets future rows
change both the mean and the direction. `aligned_scores` removes arbitrary sign
flips before comparing values; it is a diagnostic, not information used by a policy.

**Read the output:** the first two panels have identical axis limits, the same blue
early rows, different orange later rows, and each global PC1 line. The third plots
the resulting change in each protected score. The frozen line remains zero; the
global line moves. Scores are centered projections, not raw positions on the page.

**Predict, change, explain:** change the later rotation or move its mean instead.
Which fit is allowed to change earlier scores? Explain why even preprocessing without
labels must respect time. Zero change verifies this particular cutoff construction;
it does not prove every downstream pipeline is free of leakage.

In [ ]:
pca_rng = np.random.default_rng(706)

def rotated_cloud(n, angle_degrees):
    base_cloud = pca_rng.standard_normal((n, 2)) @ np.diag([1.8, 0.45])
    angle = np.deg2rad(angle_degrees)
    rotation = np.array([[np.cos(angle), -np.sin(angle)],
                         [np.sin(angle),  np.cos(angle)]])
    return base_cloud @ rotation.T

early = rotated_cloud(220, 20)
later = rotated_cloud(180, 70)
later_perturbed = later @ np.array([[0, -1], [1, 0]]).T

def protected_pca_fit(all_rows, cutoff):
    # Rebuild the pipeline from a supplied history, respecting the cutoff.
    model = PCA(n_components=1).fit(all_rows[:cutoff])
    return model, model.transform(all_rows[:cutoff])[:, 0]

early_fit, frozen_scores_before = protected_pca_fit(np.vstack([early, later]), len(early))
_, frozen_scores_after = protected_pca_fit(np.vstack([early, later_perturbed]), len(early))
global_before = PCA(n_components=1).fit(np.vstack([early, later]))
global_after = PCA(n_components=1).fit(np.vstack([early, later_perturbed]))

def aligned_scores(model, observations, reference):
    scores = model.transform(observations)[:, 0]
    return scores if np.corrcoef(scores, reference)[0, 1] >= 0 else -scores

global_scores_before = aligned_scores(global_before, early, frozen_scores_before)
global_scores_after = aligned_scores(global_after, early, frozen_scores_before)
print("largest protected-score change, frozen fit:",
      f"{np.max(np.abs(frozen_scores_after - frozen_scores_before)):.3e}")
print("largest protected-score change, global refit:",
      f"{np.max(np.abs(global_scores_after - global_scores_before)):.3f}")

fig, axes = plt.subplots(1, 3, figsize=(12.6, 3.8))
axes[0].scatter(early[:, 0], early[:, 1], s=12, alpha=.35, label="early")
axes[0].scatter(later[:, 0], later[:, 1], s=12, alpha=.25, label="later")
axes[0].set_title("original later covariance")
axes[1].scatter(early[:, 0], early[:, 1], s=12, alpha=.35, label="same early rows")
axes[1].scatter(later_perturbed[:, 0], later_perturbed[:, 1], s=12, alpha=.25,
                label="changed later covariance")
axes[1].set_title("only the future block changed")
for ax, model in zip(axes[:2], [global_before, global_after]):
    direction = model.components_[0]
    ends = model.mean_ + np.array([-3, 3])[:, None] * direction
    ax.plot(ends[:, 0], ends[:, 1], color="crimson", lw=2, label="global PC1 axis")
    ax.set(xlabel="feature 1", ylabel="feature 2", xlim=(-7, 7), ylim=(-5, 5))
    ax.legend(frameon=False, fontsize=7); ax.set_aspect("equal", adjustable="box")
axes[2].plot(global_scores_after - global_scores_before, color="crimson", lw=.8, label="global refit")
axes[2].axhline(0, color="navy", lw=2, label="frozen fit: zero change")
axes[2].set(xlabel="protected early row", ylabel="score change", title="Past scores under the two rules")
axes[2].legend(frameon=False, fontsize=8)
fig.suptitle("Demo 6: only the future block changed")
fig.tight_layout(rect=(0, 0, 1, .91))
save_lecture_figure(fig, "demo6_pca_future_invariance.png")
plt.show()

### Real-data companion — PCA on twelve industry portfolios

> **Optional empirical comparison:** Run this after the known-truth demo. Use the same transformation and compare the simulation with the historical series. Record one important difference and the most likely reason for it.

### Industry PCA: freeze before looking later

**Question:** does a training-period direction resemble broad industry movement later?
The bundled French file contains daily percentage returns. Division by 100 converts
them to decimals; invalid sentinels are removed. The teaching sample is fixed at
1990–April 2026. The first 70% of dates fit the means, scales, and PCA directions;
`transform` carries those frozen quantities to the last 30%.

This is correlation PCA: each column is divided by its training standard deviation.
`components_[0]` has 12 entries and the holdout score has one value per later date.
Its arbitrary sign is fixed from the training loadings, before inspecting holdout
returns. The bars show training variance shares and loadings; the printed correlation
uses frozen-test scores and equal-weight returns from those same 12 industries.

**Predict, change, explain:** fit covariance PCA on the same training dates and compare
its loadings. Why can a high same-panel correlation support “broad common movement”
without proving an external market factor, a causal factor, or a return forecast?
The source is a current-vintage research dataset, not archived point-in-time releases.

In [ ]:
industry_pca_raw, industry_pca_source = course_csv("datasets/famafrench/ff_12industry_daily.csv")
industry_pca_raw["date"] = pd.to_datetime(
    industry_pca_raw["date"].astype(str), format="%Y%m%d"
)
industry_pca_returns = industry_pca_raw.set_index("date").sort_index().loc["1990":"2026-04-30"] / 100.0
industry_pca_returns = industry_pca_returns.mask(industry_pca_returns <= -.999).dropna()
industry_pca_cut = int(0.7 * len(industry_pca_returns))
industry_pca_train = industry_pca_returns.iloc[:industry_pca_cut]
industry_pca_test = industry_pca_returns.iloc[industry_pca_cut:]
industry_pca_mean = industry_pca_train.mean()
industry_pca_scale = industry_pca_train.std()
industry_pca_model = PCA().fit((industry_pca_train - industry_pca_mean) / industry_pca_scale)
industry_pca_score = industry_pca_model.transform(
    (industry_pca_test - industry_pca_mean) / industry_pca_scale
)[:, 0]
industry_pca_loading = industry_pca_model.components_[0].copy()
industry_equal_weight = industry_pca_test.mean(axis=1).to_numpy()
# Fix the arbitrary sign from training loadings, before observing held-out returns.
if industry_pca_loading.sum() < 0:
    industry_pca_score *= -1
    industry_pca_loading *= -1
industry_market_correlation = np.corrcoef(industry_pca_score, industry_equal_weight)[0, 1]

print("Source: Kenneth R. French Data Library, bundled 12 industry portfolios")
print(f"File: {industry_pca_source}")
print(f"Train ends {industry_pca_train.index.max().date()}; frozen test ends {industry_pca_test.index.max().date()}")
print("Units: decimal daily returns, standardized with training-period means and scales")
print("First five explained-variance shares:", np.round(industry_pca_model.explained_variance_ratio_[:5], 3))
print(f"Frozen-test corr(PC1 score, equal-weight industry return): {industry_market_correlation:+.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar(np.arange(1, 13), industry_pca_model.explained_variance_ratio_)
axes[0].set(xlabel="component", ylabel="training variance share", title="Real industry scree plot")
axes[1].bar(industry_pca_returns.columns, industry_pca_loading)
axes[1].tick_params(axis="x", rotation=60)
axes[1].set(title="PC1 loads broadly across industries", ylabel="loading")
fig.suptitle("Real-data companion: training-period PCA on industry returns")
fig.tight_layout(rect=(0, 0, 1, .91))
save_lecture_figure(fig, "pca_industry_companion.png")
plt.show()
print("The market-like component is an interpretation supported by the loadings and score correlation, not a label supplied to PCA.")

### Demo 7 — Marchenko-Pastur eigenvalue band

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

### 7. How large can a random-data component look?

**Question:** does an uneven spectrum imply a real common factor? Draw a 300-by-80
matrix of independent normal noise. Center and scale each column; `X.T @ X / T`
is its sample correlation matrix with the code's population-variance normalization.
`eigvalsh` returns real eigenvalues for this symmetric matrix; reversing the sorted
array puts the largest first. Their sum is 80.

With `q=N/T`, the reference edges are `(1-sqrt(q))**2` and `(1+sqrt(q))**2`.
For q≈.267 these are about .234 and 2.299. This is an asymptotic benchmark under
independent unit-variance noise, not a finite-sample confidence interval. A few noise
eigenvalues may cross an edge; serial dependence and heavy tails change the null.

The second data construction adds `outer(mkt,loadings)` to **every** series and
standardizes again. Its leading eigenvalue is plotted against the original noise
spectrum on a logarithmic vertical axis. It is not a mechanical count of true factors.

**Predict, change, explain:** increase the number of series while holding observations
fixed. Predict the upper noise edge. A larger leading eigenvalue alone need not mean
stronger signal. Detection, loading accuracy, stability, and usefulness remain separate.

In [ ]:
N, T = 80, 300
X = rng.standard_normal((T, N))      # pure noise
X = (X - X.mean(0)) / X.std(0)
C = (X.T @ X) / T                    # sample correlation
w = np.sort(np.linalg.eigvalsh(C))[::-1]
q = N / T
lam_plus = (1 + np.sqrt(q)) ** 2
lam_minus = (1 - np.sqrt(q)) ** 2
print(f"MP band: [{lam_minus:.3f}, {lam_plus:.3f}]")
print(f"largest empirical eigenvalue: {w[0]:.3f}")
print(f"fraction above lam_+: {(w > lam_plus).mean():.2%}")

# Add a single planted factor by replacing one return direction
mkt = rng.standard_normal(T)
loadings = rng.uniform(0.5, 1.0, N)
Y = X + 3.0 * np.outer(mkt, loadings)
Y = (Y - Y.mean(0)) / Y.std(0)
C2 = (Y.T @ Y) / T
w2 = np.sort(np.linalg.eigvalsh(C2))[::-1]
print(f"with planted factor, largest eigenvalue: {w2[0]:.2f}  (towers above MP)")
fig, ax = plt.subplots(figsize=(8, 4))
component = np.arange(1, N + 1)
ax.plot(component, w, "o", ms=3.2, label="pure-noise eigenvalues")
ax.plot(1, w2[0], "o", ms=6.5, label="largest eigenvalue with planted factor")
ax.axhspan(lam_minus, lam_plus, color="#8da0cb", alpha=.18,
           label="Marchenko–Pastur noise band")
ax.axhline(lam_plus, color="#52658f", lw=1.2)
ax.set(xlabel="ordered component", ylabel="sample eigenvalue", yscale="log",
       title="Demo 7: noise stays in the benchmark band; the planted factor separates")
ax.legend(frameon=False)
ax.grid(alpha=.18, which="both")
fig.tight_layout()
save_lecture_figure(fig, "demo7_marchenko_pastur.png")
plt.show()
# This asymptotic band is not a finite-sample confidence interval. Some noise
# eigenvalues can cross an edge. Here the seeded noise falls inside it; a planted
#           factor produces an eigenvalue ~66, far outside it.

### Mini-project: audit a PCA representation

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

Run four checks in the notebook:

1. **Demo 5:** interpret the scree plot, PC1 loadings, and market-proxy correlation.
2. **Demo 6:** change only later covariance and test whether protected earlier scores move.
3. **Demo 7:** compare pure-noise eigenvalues and a planted factor with the Marchenko--Pastur band.
4. **Real-data companion:** fit PCA before the industry holdout and interpret frozen-test PC1.

**Notebook:** run **Demos 5–7** and the **real-data PCA companion**.

**Research record:** the simulated and industry PC1 interpretations; the noise-band comparison; the cutoff and largest earlier-score change under the frozen and global fits.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 8 — Cluster a correlation matrix to recover sectors

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

### 8. Recover groups whose truth is known

**Question:** can similarity recover four planted sectors after their column order
is scrambled? Returns have shape `(800,48)`; the 48 true sector labels are used
only for evaluation. `corrcoef(ret.T)` compares assets, not dates, producing a
48-by-48 correlation matrix. `sqrt(2*(1-corr))` is correlation distance: correlations
1, 0, and −1 give distances 0, sqrt(2), and 2. The diagonal is explicitly zero.

`AgglomerativeClustering(metric='precomputed', linkage='average', n_clusters=4)`
merges groups using average pairwise distance until four remain. `argsort(labels_)`
reorders rows and columns of the same matrix; it does not change correlations.
Purity counts each group's largest true-sector membership and divides their sum by 48.
Cluster numbers are arbitrary. Purity can improve simply by asking for more groups,
so keep the declared count fixed for this check.

**Read the output:** both heatmaps show asset pairs, with a shared correlation color
scale. Strong diagonal blocks and 100% purity describe this clean construction.

**Predict, change, explain:** weaken the sector factor before rerunning. The algorithm
still returns four groups. Explain why a partition alone does not prove persistent
sectors. Real asset clusters also need stability and a test of their intended use.

In [ ]:
n_per = 12; n_sec = 4; T = 800
demo8_rng = np.random.default_rng(42)
sector_factors = demo8_rng.standard_normal((T, n_sec)) * 0.01
market = demo8_rng.standard_normal(T) * 0.008
ret = np.zeros((T, n_per * n_sec))
sectors = np.repeat(np.arange(n_sec), n_per)
for i, s in enumerate(sectors):
    ret[:, i] = 0.6 * market + 1.2 * sector_factors[:, s] + 0.01 * demo8_rng.standard_normal(T)
# scramble columns
perm = demo8_rng.permutation(ret.shape[1])
ret = ret[:, perm]
true_labels = sectors[perm]

corr = np.corrcoef(ret.T)
dist = np.sqrt(2 * np.clip(1 - corr, 0, None))
np.fill_diagonal(dist, 0)
clust = AgglomerativeClustering(n_clusters=4, metric="precomputed", linkage="average").fit(dist)

# purity
purity = 0
for k in range(4):
    members = np.where(clust.labels_ == k)[0]
    if len(members):
        most_common = np.bincount(true_labels[members]).max()
        purity += most_common
purity = purity / len(true_labels)
print(f"Cluster purity: {purity:.1%}")

order = np.argsort(clust.labels_)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
image = axes[0].imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
axes[0].set_title("Raw (scrambled) correlation")
axes[1].imshow(corr[order][:, order], cmap="RdBu_r", vmin=-1, vmax=1)
axes[1].set_title("Reordered by recovered clusters")
for ax in axes:
    ax.set_xlabel("asset order")
    ax.set_ylabel("asset order")
fig.suptitle(f"Demo 8: four planted sectors recovered (purity {purity:.0%})")
fig.subplots_adjust(top=.84, right=.88, wspace=.24)
color_axis = fig.add_axes([.91, .20, .018, .55])
fig.colorbar(image, cax=color_axis, label="correlation")
save_lecture_figure(fig, "demo8_sector_recovery.png")
plt.show()
# Expected: cluster purity ~100% on clean simulated sectors. Real correlation matrices
#           do not behave this well.

### Demo 9 — Frozen-centroid regime assignment

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

### 9. Label a date using what was known then

**Question:** do completed-history labels match labels from a frozen earlier rule?
`sim.regime_switching` generates 1,500 returns. Each feature row contains a trailing
21-return mean and standard deviation. `.shift(1)` makes the row at index t use
returns t−21 through t−1, so it is available before return t.

Rows below index 900 fit `StandardScaler` and two K-means centers. K-means alternates
nearest-center assignment and averaging; `n_init=20` tries 20 starts and keeps its
best within-cluster objective. `predict` assigns later rows without refitting.
`inverse_transform` expresses centers in return units; the larger volatility center
gets the descriptive label “high.” That naming rule uses training centers only.

A second fit sees the whole feature history. Both versions use lagged features, so
this comparison isolates future influence through fitting and scaling, not through
a centered future-return window. The figure overlays the two later label sequences
and shades disagreement; the printed transition count is not accuracy against the
hidden true state. The simulator output here contains returns, not true labels.

**Predict, change, explain:** alter the cutoff or rolling window, declared before
running. Why can most labels agree while the full-sample rule is still inadmissible
for historical decisions? A valid clock does not establish profitability or correct
state recovery. Never rename a training group using later returns.

In [ ]:
r = sim.regime_switching(n=1500, sig_calm=0.006, sig_turb=0.02,
                         p_stay_calm=0.985, p_stay_turb=0.96, seed=11)
r = np.asarray(r)

window = 21
series = pd.Series(r)
features = pd.DataFrame({
    "lagged_mean": series.rolling(window).mean().shift(1),
    "lagged_vol": series.rolling(window).std().shift(1),
}).dropna()
cutoff = 900
train = features.loc[features.index < cutoff]
test = features.loc[features.index >= cutoff]

train_scaler = StandardScaler().fit(train)
frozen_model = KMeans(n_clusters=2, n_init=20, random_state=11).fit(
    train_scaler.transform(train)
)
frozen_raw_centroids = train_scaler.inverse_transform(frozen_model.cluster_centers_)
frozen_high = np.argmax(frozen_raw_centroids[:, 1])
frozen_labels = (frozen_model.predict(train_scaler.transform(test)) == frozen_high).astype(int)

full_scaler = StandardScaler().fit(features)
full_model = KMeans(n_clusters=2, n_init=20, random_state=11).fit(
    full_scaler.transform(features)
)
full_raw_centroids = full_scaler.inverse_transform(full_model.cluster_centers_)
full_high = np.argmax(full_raw_centroids[:, 1])
retrospective_labels = (
    full_model.predict(full_scaler.transform(test)) == full_high
).astype(int)

disagree = np.mean(frozen_labels != retrospective_labels)
transitions = np.sum(frozen_labels[1:] != frozen_labels[:-1])
print("training cutoff index:", cutoff)
print("frozen centroids [lagged mean, lagged vol]:\n", frozen_raw_centroids.round(4))
print(f"test-label disagreement with full-sample clustering: {disagree:.1%}")
print(f"transitions under the frozen rule: {transitions}")

fig, axes = plt.subplots(2, 1, figsize=(10, 4.8), sharex=True)
axes[0].plot(test.index, test["lagged_vol"], color="black", lw=.8)
axes[0].set(ylabel="lagged volatility", title="Both label series describe the same later rows")
axes[1].step(test.index, frozen_labels, where="post", label="frozen-centroid label")
axes[1].step(test.index, retrospective_labels, where="post", alpha=.7,
             label="full-sample label")
axes[1].fill_between(
    test.index, 0, 1, where=frozen_labels != retrospective_labels,
    step="post", color="#d95f02", alpha=.28, label="labels disagree",
)
axes[1].set(xlabel="time index", ylabel="high-vol label", yticks=[0, 1])
axes[1].legend(frameon=False, ncol=2)
fig.suptitle(f"Demo 9: frozen and full-sample labels disagree on {disagree:.1%} of later dates")
fig.tight_layout(rect=(0, 0, 1, .94))
save_lecture_figure(fig, "demo9_frozen_centroid_assignments.png")
plt.show()

### Mini-project: recover groups, then make a regime prospective

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

1. **Demo 8:** cluster the scrambled correlation matrix and record sector recovery.
2. **Demo 9:** freeze earlier centroids, assign later dates, and compare with
   full-sample labels.

**Notebook:** run **Demos 8 and 9**.

**Research record:** Demo 8 cluster purity and reordered heatmap; Demo 9 training cutoff, frozen centroids, label disagreement, and transition timing.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Optional example — the yield curve, where the components have names

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

### Optional Treasury example: changes and levels answer different questions

**Question:** do Treasury yields move mostly through a few recognizable shapes?
The seven columns are 1–7-year zero-coupon yields in percentage points. The bundled
Fed snapshot ends April 2026. `curve.diff()` makes daily changes; `np.cov(...,
rowvar=False)` estimates their 7-by-7 covariance. `eigh` returns directions and
variances, sorted from largest to smallest. A change of .01 percentage point is
one basis point.

The directions are learned from **changes**, while `states=(curve-curve.mean()) @ V`
projects **levels** onto those same directions. These are different quantities:
a time-varying score does not mean a time-varying loading. Signs are oriented against
level, slope, and belly-versus-wings references for this retrospective description.
Names must be supported by the loading shapes; PCA was not given those names.

The last panel compares absolute dot products of corresponding unit directions
from two sample halves: 1 means agreement up to sign. This single split is a limited
stability check; nearly tied components may rotate while their joint span is stable.
The full-history fit and centering do not supply a historical trading signal.

**Predict, change, explain:** compare PCA fitted to levels instead of changes. Why
would it answer a different risk question? This compact seven-tenor cell does not
reproduce the separate eleven-tenor addendum's snapshots and shock charts; their
builders and sources are linked in the Week 7 reading guide.

In [ ]:
# Demos 5 to 7 fitted PCA to simulated and industry panels, where the components
# were real but anonymous. On the Treasury curve the leading directions have
# conventional names. The point of this cell is the last line of its output: the
# names are a claim about stability, not a property of the algebra.
from urllib.request import Request


def gsw_curve(maturities=range(1, 8)):
    """Read the bundled zero-coupon curve, or its public Fed source in Colab."""
    relative = "datasets/fred/nominal_zero_coupon_curve_1_7y.csv"
    for root in (Path.cwd(), *Path.cwd().parents):
        local = root / relative
        if local.exists():
            frame = pd.read_csv(local, index_col="date", parse_dates=True)
            return frame, str(local)
    # Gürkaynak-Sack-Wright daily zero-coupon yields; the file carries nine
    # header lines of documentation before the column row.
    url = "https://www.federalreserve.gov/data/yield-curve-tables/feds200628.csv"
    request = Request(url, headers={"User-Agent": "Mozilla/5.0 finml-course"})
    raw = pd.read_csv(BytesIO(urlopen(request, timeout=90).read()), skiprows=9,
                      parse_dates=["Date"], na_values=["NA", -999.99]).set_index("Date")
    columns = [f"SVENY{m:02d}" for m in maturities]
    frame = raw[columns].rename(columns={c: f"{int(c[-2:])}y" for c in columns}).dropna()
    frame.index.name = "date"
    return frame, url


curve, curve_source = gsw_curve()
curve = curve.loc[:"2026-04-30"]  # fixed teaching snapshot for live fallback too
labels = list(curve.columns)

# Fit PCA to daily *changes*: a risk model is about how the curve moves. Then
# project centered *levels* onto those fixed loadings to read the factor states.
changes = curve.diff().dropna()
centered_changes = changes - changes.mean()
eigenvalues, eigenvectors = eigh(np.cov(centered_changes.to_numpy(), rowvar=False))
order = np.argsort(eigenvalues)[::-1]
eigenvalues, eigenvectors = eigenvalues[order], eigenvectors[:, order]
shares = eigenvalues / eigenvalues.sum()
states = (curve - curve.mean()).to_numpy() @ eigenvectors[:, :3]

# An eigenvector's sign is arbitrary, so fix it against an economic reference
# before naming anything. These are conventions, not discoveries.
references = [curve.mean(axis=1).to_numpy(),
              (curve["7y"] - curve["1y"]).to_numpy(),
              (curve["4y"] - 0.5 * (curve["1y"] + curve["7y"])).to_numpy()]
for k, reference in enumerate(references):
    if np.corrcoef(states[:, k], reference)[0, 1] < 0:
        eigenvectors[:, k], states[:, k] = -eigenvectors[:, k], -states[:, k]

print(f"Source: Federal Reserve Board, Gürkaynak-Sack-Wright zero-coupon curve")
print(f"File: {curve_source}")
print(f"Sample: {curve.index.min():%Y-%m-%d} to {curve.index.max():%Y-%m-%d}, "
      f"{len(curve):,} daily observations of 1-7y yields")
print(f"Variance shares of daily changes: PC1 {shares[0]:.3f}  PC2 {shares[1]:.3f}  "
      f"PC3 {shares[2]:.3f}  (first three {shares[:3].sum():.3f})")
for name, vector in zip(["level  ", "slope  ", "curvature"], eigenvectors[:, :3].T):
    print(f"  {name} loadings: " + "  ".join(f"{label} {w:+.2f}" for label, w in zip(labels, vector)))

# Does the naming survive being refitted on a different era? Split the sample
# and compare each loading vector, after sign alignment, by its overlap.
half = len(changes) // 2
overlaps = []
for piece in (changes.iloc[:half], changes.iloc[half:]):
    centered = piece - piece.mean()
    values, vectors = eigh(np.cov(centered.to_numpy(), rowvar=False))
    vectors = vectors[:, np.argsort(values)[::-1]]
    overlaps.append(vectors[:, :3])
era_overlap = [abs(float(overlaps[0][:, k] @ overlaps[1][:, k])) for k in range(3)]

fig, axes = plt.subplots(1, 3, figsize=(12.6, 3.6))
for k, (name, color) in enumerate(zip(["PC1 level", "PC2 slope", "PC3 curvature"],
                                      ["#285a84", "#9f3a2f", "#3f7754"])):
    axes[0].plot(np.arange(1, len(labels) + 1), eigenvectors[:, k], marker="o", ms=4,
                 color=color, label=f"{name} ({shares[k]:.1%})")
axes[0].axhline(0, color="0.6", lw=.8)
axes[0].set(xlabel="maturity (years)", ylabel="loading", xticks=np.arange(1, len(labels) + 1))
axes[0].set_title("A. Loadings earn the names", fontsize=10)
axes[0].legend(frameon=False, fontsize=8)

monthly = pd.DataFrame(states, index=curve.index, columns=["PC1", "PC2", "PC3"]).resample("ME").last()
for column, color in zip(monthly.columns, ["#285a84", "#9f3a2f", "#3f7754"]):
    axes[1].plot(monthly.index, monthly[column], lw=.8, color=color, label=column)
axes[1].axhline(0, color="0.6", lw=.8)
axes[1].set(xlabel="date", ylabel="factor state")
axes[1].set_title("B. Historical levels in fixed coordinates", fontsize=10)
axes[1].legend(frameon=False, fontsize=8, ncol=3)

axes[2].bar(["PC1", "PC2", "PC3"], era_overlap, color=["#285a84", "#9f3a2f", "#3f7754"])
axes[2].axhline(1.0, color="0.6", lw=.8, ls=":")
axes[2].set(ylim=(0, 1.08), ylabel="|overlap| across halves")
axes[2].set_title("C. Do the directions survive a refit?", fontsize=10)
for spine in ("top", "right"):
    for axis in axes:
        axis.spines[spine].set_visible(False)
fig.tight_layout()
save_lecture_figure(fig, "optional_yield_curve_pca.png")
plt.show()

print("Loading overlap between the two sample halves: " +
      "  ".join(f"PC{k + 1} {value:.3f}" for k, value in enumerate(era_overlap)))
print("The names are supported by the loading shapes and checked by the refit, not supplied by PCA.")
print("This full-sample basis and centering describe history; they are not a live backtest.")
print("The 1–7y notebook example differs from the 11-tenor addendum; it does not reproduce all addendum figures.")
print("A direction that explains variance is not automatically a tradable factor:")
print("translating these into P&L needs key-rate durations, which the research note covers.")
# Expected: PC1 nearly flat and dominant (about 92% of change variance), PC2
#           rising monotonically across maturity, PC3 a belly-versus-wings
#           shape. All three overlaps come out above .95, so here the naming
#           does survive a refit. That is the favorable case; compare it with
#           the industry panel in the Deck B companion, where the components
#           are equally real and have no names to earn.

## Where this leaves us

Optimization can converge perfectly to an unstable answer, and an attractive component or cluster can still be sample-dependent. These tools simplify or organize evidence; they do not interpret themselves and do not automatically create a decision rule.

Week 8 puts estimated signals into portfolio decisions and then studies how neural networks build flexible representations from data.